In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import numpy as np
import pandas as pd
import time 
import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import (SelectKBest, f_regression, RFE, SequentialFeatureSelector)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv('ICEV_modeling_ready.csv')

In [4]:
df = df.drop(columns={'Unnamed: 0'})
df = df.dropna()

In [5]:
df["delta_time_sec"] = (df.groupby(["vehid", "trip"])["timestamp"].diff().fillna(0).div(1000))
df["accel"] = (df.groupby(["vehid", "trip"])["speed_kmh"].diff()/ df["delta_time_sec"]).fillna(0)

# Rolling speed (indicates if there's congestion, aggressive driving, transient engine load)
df["speed_roll_mean_10"] = df.groupby(["vehid", "trip"])["speed_kmh"].rolling(10).mean().reset_index(level=[0,1], drop=True)
df["accel_roll_std_10"] = df.groupby(["vehid", "trip"])["accel"].rolling(10).std().reset_index(level=[0,1], drop=True)

# encode cyclical time
df["day_sin"] = np.sin(2*np.pi*df["day_of_week"]/7)
df["day_cos"] = np.cos(2*np.pi*df["day_of_week"]/7)

# Forward-fill within trip for null speed_roll_mean_10 and accel_roll_std_10
df = df.sort_values(["vehid", "trip", "timestamp"])
df["speed_roll_mean_10"] = df.groupby(["vehid", "trip"])["speed_kmh"].rolling(10, min_periods=1).mean().reset_index(level=[0,1], drop=True)
df["accel_roll_std_10"] = df.groupby(["vehid", "trip"])["accel"].rolling(10, min_periods=1).std().reset_index(level=[0,1], drop=True)

# Fill the rest of NaN with instant values
df["speed_roll_mean_10"] = df["speed_roll_mean_10"].fillna(df["speed_kmh"])
df["accel_roll_std_10"] = df["accel_roll_std_10"].fillna(0)

In [6]:
TARGET = "encon"
TEMPORAL_COL = "timestamp"

In [7]:
NUMERIC_COLS = ["speed_kmh", "temp_degc", "displacement", "cylinders", "elapsed_min", "accel", "speed_roll_mean_10", "accel_roll_std_10", "day_sin", "day_cos"]
CATEGORICAL_COLS = ["inferred_class", "transmission", "road_class"]
ALL_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

In [8]:
df_encoded = pd.get_dummies(df, columns=CATEGORICAL_COLS, drop_first=True)
NEW_CATEGORICAL_COLS = [col for col in df_encoded.columns if any(orig in col for orig in CATEGORICAL_COLS)]
ALL_FEATURES = NUMERIC_COLS + NEW_CATEGORICAL_COLS

In [9]:
# Get all unique combinations of (vehid, trip)
unique_trips = df_encoded[['vehid', 'trip']].drop_duplicates()
unique_trips.shape

(17660, 2)

In [10]:
# Sample trip
sampled_trips = unique_trips.sample(n=5000, random_state=42)
# Filter the main dataframe using merge
df_sampled = df_encoded.merge(sampled_trips, on=['vehid', 'trip'], how='inner')

In [11]:
print(f"Original shape: {df_encoded.shape}")
print(f"Sampled shape: {df_sampled.shape}")
print(f"Unique trips kept: {df_sampled.groupby(['vehid', 'trip']).ngroups}")

Original shape: (12493093, 22)
Sampled shape: (3503143, 22)
Unique trips kept: 5000


In [12]:
def temporal_split(df, train_frac=0.98, dev_frac=0.01):
    df_sorted = df.sort_values(TEMPORAL_COL).reset_index(drop=True)
    n = len(df_sorted)
    train_end = int(n * train_frac)
    dev_end = int(n * (train_frac + dev_frac))

    train = df_sorted.iloc[:train_end]
    dev = df_sorted.iloc[train_end:dev_end]
    test = df_sorted.iloc[dev_end:]

    for split, name in [(train, "Train"), (dev, "Dev"), (test, "Test")]:
        print(f"{name}: {len(split)} rows from "
              f"{split[TEMPORAL_COL].min()} to {split[TEMPORAL_COL].max()}")
    return train, dev, test

In [13]:
train, dev, test = temporal_split(df_sampled)

Train: 3433080 rows from 0 to 1789000
Dev: 35031 rows from 1789000 to 2286000
Test: 35032 rows from 2286100 to 5734700


In [14]:
def scale_features(train, dev, test, numeric_cols, all_cols):
    scaler  = StandardScaler()
    num_idx = [all_cols.index(c) for c in numeric_cols]

    def to_float(df):
        return df[all_cols].values.astype(float)

    X_tr = to_float(train)
    X_dv = to_float(dev)
    X_te = to_float(test)

    X_tr[:, num_idx] = scaler.fit_transform(X_tr[:, num_idx])
    X_dv[:, num_idx] = scaler.transform(X_dv[:, num_idx])
    X_te[:, num_idx] = scaler.transform(X_te[:, num_idx])

    return X_tr, X_dv, X_te, scaler

In [15]:
X_tr, X_dv, X_te, scaler = scale_features(train, dev, test, NUMERIC_COLS, ALL_FEATURES)

y_tr = train[TARGET].values
y_dv = dev[TARGET].values
y_te = test[TARGET].values
print("Feature Matrix:")
print(f"train: {X_tr.shape}") 
print(f"dev: {X_dv.shape}")
print(f"test: {X_te.shape}")
print("Target Range:")
print(f"train: [{y_tr.min():.3f}, {y_tr.max():.3f}]", f"mean={y_tr.mean():.3f}")

Feature Matrix:
train: (3433080, 16)
dev: (35031, 16)
test: (35032, 16)
Target Range:
train: [0.000, 25.300] mean=0.874


In [16]:
SEQ_LEN = 20       # look-back window (rows).  Tune: 10 / 20 / 50
LSTM_UNITS = [64, 32]
DENSE_UNITS = 32
DROPOUT = 0.1
BATCH_SIZE = 2048
EPOCHS = 50
PATIENCE_ES = 7
PATIENCE_LR = 4
LR = 1e-3

In [16]:
# Creates sequences => ensuring windows do not cross boundaries of different vehicles or trips using strict sequential indices.
def create_sequences_by_trip(df, feature_matrix, target_array, seq_len=20):
    X_seq, y_seq = [], []

    # Map absolute integer positions to rows
    df_pos = df.copy()
    df_pos['row_pos'] = np.arange(len(df))

    grouped = df_pos.groupby(["vehid", "trip"])

    for _, group in grouped:
        positions = group['row_pos'].values

        if len(positions) < seq_len:
            continue
        for i in range(len(positions) - seq_len):
            # Extract exactly seq_len consecutive rows belonging to this trip
            trip_slice_indices = positions[i : i + seq_len]

            # Slice feature matrix using the exact block of indices
            X_seq.append(feature_matrix[trip_slice_indices])

            # Target is the value immediately following the sequence
            target_pos = positions[i + seq_len]
            y_seq.append(target_array[target_pos])

    return np.array(X_seq), np.array(y_seq)

In [18]:
X_tr_seq, y_tr_seq = create_sequences_by_trip(train, X_tr, y_tr, SEQ_LEN)
X_dv_seq, y_dv_seq = create_sequences_by_trip(dev, X_dv, y_dv, SEQ_LEN)
X_te_seq, y_te_seq = create_sequences_by_trip(test, X_te, y_te, SEQ_LEN)

print(f"Train seq shape: {X_tr_seq.shape}")
print(f"Dev seq shape: {X_dv_seq.shape}")
print(f"Test seq shape: {X_te_seq.shape}")

Train seq shape: (3333087, 20, 16)
Dev seq shape: (33194, 20, 16)
Test seq shape: (34092, 20, 16)


In [19]:
model = Sequential([
    LSTM(64,input_shape=(X_tr_seq.shape[1], X_tr_seq.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)])

model.compile(optimizer="adam", loss="mse")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          20,736 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22,849 (89.25 KB)

 Trainable params: 22,849 (89.25 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(X_tr_seq, y_tr_seq,
    validation_data=(X_dv_seq, y_dv_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_LR, min_lr=1e-6)],
    verbose=1)

Epoch 1/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 81s 49ms/step - loss: 0.3677 - val_loss: 0.3590 - learning_rate: 0.0010
Epoch 2/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 79s 49ms/step - loss: 0.3292 - val_loss: 0.3487 - learning_rate: 0.0010
Epoch 3/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3217 - val_loss: 0.3429 - learning_rate: 0.0010
Epoch 4/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3180 - val_loss: 0.3550 - learning_rate: 0.0010
Epoch 5/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3144 - val_loss: 0.3532 - learning_rate: 0.0010
Epoch 6/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3125 - val_loss: 0.3505 - learning_rate: 0.0010
Epoch 7/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3094 - val_loss: 0.3520 - learning_rate: 0.0010
Epoch 8/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3054 - val_loss: 0.3561 - learning_rate: 5.0000e-04
Epoch 9/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 80s 49ms/step - loss: 0.3037 - val_loss: 0

In [21]:
dev_pred = model.predict(X_dv_seq).flatten()

1038/1038 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [23]:
def evaluate(name, y_true, y_pred, split="dev"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    mask = np.abs(y_true) > 0.01 # skip near-zero rows
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else float("nan")

    print(f"{name} on Dev Set")
    print(f"MAE: {mae:.4f} L/hr")
    print(f"RMSE: {rmse:.4f} L/hr")
    print(f"R-squared: {r2:.4f}")
    print(f"MAPE: {mape:.2f}%")

    return {"model": name, "split": split,  "MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

In [24]:
result = evaluate("LSTM", y_dv_seq, dev_pred)

LSTM on Dev Set
MAE: 0.3879 L/hr
RMSE: 0.5856 L/hr
R-squared: 0.4924
MAPE: 59.83%


## No Early Stopping

In [25]:
# Add this cell right after defining your model and before running model.predict()
history = model.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_dv_seq, y_dv_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_LR, min_lr=1e-6)],
    verbose=1)

Epoch 1/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 81s 50ms/step - loss: 0.3161 - val_loss: 0.3563 - learning_rate: 5.0000e-04
Epoch 2/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 83s 51ms/step - loss: 0.3142 - val_loss: 0.3528 - learning_rate: 5.0000e-04
Epoch 3/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - loss: 0.3127 - val_loss: 0.3484 - learning_rate: 5.0000e-04
Epoch 4/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - loss: 0.3110 - val_loss: 0.3582 - learning_rate: 5.0000e-04
Epoch 5/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - loss: 0.3098 - val_loss: 0.3584 - learning_rate: 5.0000e-04
Epoch 6/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - loss: 0.3082 - val_loss: 0.3582 - learning_rate: 5.0000e-04
Epoch 7/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 84s 52ms/step - loss: 0.3072 - val_loss: 0.3630 - learning_rate: 5.0000e-04
Epoch 8/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - loss: 0.3046 - val_loss: 0.3583 - learning_rate: 2.5000e-04
Epoch 9/50
1628/1628 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step 

In [26]:
dev_pred = model.predict(X_dv_seq).flatten()
result = evaluate("LSTM", y_dv_seq, dev_pred)

1038/1038 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
LSTM on Dev Set
MAE: 0.3984 L/hr
RMSE: 0.6041 L/hr
R-squared: 0.4598
MAPE: 59.06%


In [27]:
train_pred = model.predict(X_tr_seq).flatten()
result = evaluate("LSTM", y_tr_seq, train_pred)

104159/104159 ━━━━━━━━━━━━━━━━━━━━ 178s 2ms/step
LSTM on Dev Set
MAE: 0.3653 L/hr
RMSE: 0.5417 L/hr
R-squared: 0.5547
MAPE: 60.37%


In [28]:
train

,vehid,trip,timestamp,speed_kmh,temp_degc,displacement,cylinders,day_of_week,elapsed_min,encon,...,speed_roll_mean_10,accel_roll_std_10,day_sin,day_cos,inferred_class_Large,inferred_class_Medium,inferred_class_Small,road_class_Main,road_class_Residential,road_class_Unclassified
0,530,563,0,41.0,-0.823497,2.5,4,0,0.000000,0.202692,...,41.0,0.000000,0.000000,1.000000,False,True,False,False,True,False
1,507,1479,0,0.0,15.000000,2.4,4,2,0.000000,1.228693,...,0.0,0.000000,0.974928,-0.222521,False,True,False,False,False,False
2,223,1217,0,78.0,15.000000,1.4,4,5,0.000000,0.354313,...,78.0,0.000000,-0.974928,-0.222521,False,False,True,False,False,False
3,464,1119,0,52.0,-0.823497,1.8,4,5,0.000000,1.070684,...,52.0,0.000000,-0.974928,-0.222521,False,False,True,False,True,False
4,271,1166,0,34.0,3.345260,2.0,4,3,0.000000,1.149370,...,34.0,0.000000,0.433884,-0.900969,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3433075,203,588,1789000,0.0,-1.390018,3.3,6,2,29.816667,0.206368,...,0.0,0.000000,0.974928,-0.222521,True,False,False,True,False,False
3433076,282,792,1789000,19.0,-1.390018,2.5,4,6,29.816667,0.817982,...,11.3,3.784471,-0.781831,0.623490,False,True,False,True,False,False
3433077,494,1973,1789000,50.0,7.230571,3.5,6,5,29.816667,0.285395,...,49.0,3.990730,-0.974928,-0.222521,True,False,False,True,False,False
3433078,289,1917,1789000,48.0,3.345260,1.8,4,2,29.816667,0.245533,...,50.7,1.699025,0.974928,-0.222521,False,False,True,True,False,False


# Tuning

In [29]:
SEQ_LEN = 20      
LSTM_UNITS = [64, 32]
DENSE_UNITS = 32
DROPOUT = 0.1
BATCH_SIZE = 1024
EPOCHS = 50
PATIENCE_ES = 7
PATIENCE_LR = 4
LR = 1e-3

print("Build sequences")
X_tr_seq, y_tr_seq = create_sequences_by_trip(train, X_tr, y_tr, SEQ_LEN)
X_dv_seq, y_dv_seq = create_sequences_by_trip(dev, X_dv, y_dv, SEQ_LEN)
X_te_seq, y_te_seq = create_sequences_by_trip(test, X_te, y_te, SEQ_LEN)

print(f"Train seq shape: {X_tr_seq.shape}")
print(f"Dev seq shape: {X_dv_seq.shape}")
print(f"Test seq shape: {X_te_seq.shape}")

Build sequences
Train seq shape: (3333087, 20, 16)
Dev seq shape: (33194, 20, 16)
Test seq shape: (34092, 20, 16)


In [30]:
print("Train Model")
model = Sequential([
    LSTM(64,input_shape=(X_tr_seq.shape[1], X_tr_seq.shape[2]), return_sequences=False),
    Dense(32, activation="relu"),
    Dense(1)])

optimizer = Adam(learning_rate=LR)
model.compile(optimizer=optimizer, loss='mse')
model.summary()

Train Model


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                        │ (None, 64)                  │          20,736 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22,849 (89.25 KB)

 Trainable params: 22,849 (89.25 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history = model.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_dv_seq, y_dv_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1)

Epoch 1/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 182s 56ms/step - loss: 0.3437 - val_loss: 0.4023
Epoch 2/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 177s 54ms/step - loss: 0.3182 - val_loss: 0.3507
Epoch 3/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.3119 - val_loss: 0.3525
Epoch 4/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.3056 - val_loss: 0.3737
Epoch 5/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.3003 - val_loss: 0.3665
Epoch 6/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.2963 - val_loss: 0.3592
Epoch 7/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.2938 - val_loss: 0.3541
Epoch 8/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.2907 - val_loss: 0.3706
Epoch 9/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.2887 - val_loss: 0.3637
Epoch 10/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 176s 54ms/step - loss: 0.2868 - val_loss: 0.3773
Epoch 11/50
3255/3255 ━━━━━━━━━━━━━━━━━━━━ 177s 54ms/step - loss: 0.2863 - val_loss: 0.36

In [32]:
dev_pred = model.predict(X_dv_seq).flatten()
result = evaluate("LSTM", y_dv_seq, dev_pred)

1038/1038 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
LSTM on Dev Set
MAE: 0.4437 L/hr
RMSE: 0.6712 L/hr
R-squared: 0.3331
MAPE: 66.52%


In [33]:
dev_pred = model.predict(X_tr_seq).flatten()
result = evaluate("LSTM", y_tr_seq, dev_pred)

104159/104159 ━━━━━━━━━━━━━━━━━━━━ 238s 2ms/step
LSTM on Dev Set
MAE: 0.3476 L/hr
RMSE: 0.5077 L/hr
R-squared: 0.6089
MAPE: 58.12%


# TCN

In [18]:
from tcn import TCN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [19]:
# TCN receptive field = 2^(nb_stacks * len(dilations)) * kernel_size
# With defaults below: 2^(1*8) * 3 = 768 steps — more than enough for SEQ_LEN=20
TCN_FILTERS = 64          # equivalent to LSTM units
KERNEL_SIZE = 3           # how many timesteps each conv kernel sees
DILATIONS = [1,2,4,8]   # exponential dilation — covers long-range dependencies
NB_STACKS = 1           # how many times to repeat the dilation stack
DROPOUT = 0.1         # same logic as LSTM — large data, keep low
DENSE_UNITS = 32
BATCH_SIZE = 2048
EPOCHS = 50
LR = 1e-3

In [20]:
SEQ_LEN = 20

In [21]:
X_tr_seq, y_tr_seq = create_sequences_by_trip(train, X_tr, y_tr, SEQ_LEN)
X_dv_seq, y_dv_seq = create_sequences_by_trip(dev, X_dv, y_dv, SEQ_LEN)
X_te_seq, y_te_seq = create_sequences_by_trip(test, X_te, y_te, SEQ_LEN)

print(f"Train seq shape: {X_tr_seq.shape}")
print(f"Dev seq shape: {X_dv_seq.shape}")
print(f"Test seq shape: {X_te_seq.shape}")

Train seq shape: (3333087, 20, 16)
Dev seq shape: (33194, 20, 16)
Test seq shape: (34092, 20, 16)


In [23]:
model_tcn = Sequential([
    TCN(nb_filters=TCN_FILTERS,
        kernel_size=KERNEL_SIZE,
        dilations=DILATIONS,
        nb_stacks=NB_STACKS,
        dropout_rate=DROPOUT,
        return_sequences=False,      # single vector out
        use_batch_norm=True,         # stabilises training, replaces BatchNormalization layer
        use_skip_connections=True,   # skip connections = better gradient flow, like ResNet
        padding="causal",            # no future leakage
        input_shape=(X_tr_seq.shape[1], X_tr_seq.shape[2])
    ),
    Dense(DENSE_UNITS, activation="relu"),
    Dense(1)
])

In [24]:
model_tcn.compile(
    optimizer=Adam(learning_rate=LR, clipnorm=1.0),
    loss="mse")

model_tcn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ tcn (TCN)                            │ (None, 64)                  │          92,736 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 94,849 (370.50 KB)

 Trainable params: 93,825 (366.50 KB)

 Non-trainable params: 1,024 (4.00 KB)

In [25]:
from tcn import compiled_tcn
rf = 1
for d in DILATIONS * NB_STACKS:
    rf += (KERNEL_SIZE - 1) * d
print(f"Receptive field: {rf} timesteps")
print(f"SEQ_LEN: {X_tr_seq.shape[1]} timesteps")
print(f"Check sequence length: {rf >= X_tr_seq.shape[1]}")

Receptive field: 31 timesteps
SEQ_LEN: 20 timesteps
Check sequence length: True


In [26]:
callbacks = [EarlyStopping(monitor="val_loss", patience=7,restore_best_weights=True, verbose=1),
             ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),]

In [27]:
history = model_tcn.fit(
    X_tr_seq, y_tr_seq, 
    validation_data=(X_dv_seq, y_dv_seq),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1)

Epoch 1/50
104159/104159 ━━━━━━━━━━━━━━━━━━━━ 1070s 10ms/step - loss: 0.3395 - val_loss: 0.3425 - learning_rate: 0.0010
Epoch 2/50
104159/104159 ━━━━━━━━━━━━━━━━━━━━ 1062s 10ms/step - loss: 0.3221 - val_loss: 0.3394 - learning_rate: 0.0010
Epoch 3/50
104159/104159 ━━━━━━━━━━━━━━━━━━━━ 1060s 10ms/step - loss: 0.3174 - val_loss: 0.3368 - learning_rate: 0.0010
Epoch 4/50
104159/104159 ━━━━━━━━━━━━━━━━━━━━ 1080s 10ms/step - loss: 0.3142 - val_loss: 0.3467 - learning_rate: 0.0010
Epoch 5/50
104159/104159 ━━━━━━━━━━━━━━━━━━━━ 1292s 12ms/step - loss: 0.3116 - val_loss: 0.3395 - learning_rate: 0.0010
Epoch 6/50
100060/104159 ━━━━━━━━━━━━━━━━━━━━ 1:07 16ms/step - loss: 0.3103

KeyboardInterrupt: 

In [ ]:
dev_pred_s = model_tcn.predict(X_dv_seq).flatten()

In [ ]:
result_tcn = evaluate("TCN", y_dv_seq, dev_pred_s)

In [ ]:
train_pred_s = model_tcn.predict(X_tr_seq).flatten()

In [ ]:
result_tcn = evaluate("TCN", y_tr_seq, train_pred_s)